<a href="https://colab.research.google.com/github/ishaanrai-hub/IIT-Hyd-Projects-Machine-learning-/blob/main/rare_disease_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
##Importing libraries
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

In [2]:
# Part A: Calculate k
# -----------------------------

n_samples = 1000
percentage = 0.05

k = int(n_samples * percentage)

# Make k odd to avoid ties
if k % 2 == 0:
    k += 1

print(f"Chosen value of k: {k}")

Chosen value of k: 51


In [3]:
# Create Synthetic Dataset
# -----------------------------

np.random.seed(42)

# 50 numerical medical test results (already between 0 and 1)
X = np.random.rand(1000, 50)

# Labels: 0 = no disease, 1 = rare disease
y = np.array([0] * 950 + [1] * 50)

# Shuffle dataset
indices = np.random.permutation(len(y))
X = X[indices]
y = y[indices]

In [4]:
# Train-Test Split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [5]:
# Part B: Preprocessing Step 1
# Feature Scaling (Min-Max)
# -----------------------------

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
# Part B: Preprocessing Step 2
# Distance-Weighted k-NN
# -----------------------------

knn = KNeighborsClassifier(
    n_neighbors=k,
    weights="distance",  # handles class imbalance
    metric="euclidean"
)

In [7]:
# Train model
knn.fit(X_train_scaled, y_train)

# Predictions
y_pred = knn.predict(X_test_scaled)

# -----------------------------
# Evaluation
# -----------------------------

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       190
           1       0.00      0.00      0.00        10

    accuracy                           0.95       200
   macro avg       0.47      0.50      0.49       200
weighted avg       0.90      0.95      0.93       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [8]:
# # # Why k-NN Struggles with Rare Diseases
# # # Core Problem: Class Imbalance + Majority Voting

# # # In your scenario:

# # # Total patients = 1,000

# # # Rare disease cases = 50 (5%)

# # # Non-disease cases = 950 (95%)

# # # k-NN predicts a label by majority vote among the k nearest neighbors.
# # # When one class is extremely rare, the neighborhood around a test point is overwhelmingly dominated by the majority class.
# # Why This Is Especially Bad for Medical Diagnosis
# # 1. Low Recall for Rare Diseases

# # k-NN tends to predict the majority class:

# # High overall accuracy (looks good)

# # Very low sensitivity (recall) for disease cases

# # This is dangerous in healthcare because:

# # Missing a sick patient is worse than falsely flagging a healthy one.

# # 2. Distance Does Not Fix Imbalance

# # Even with distance-weighted k-NN:

# # Rare disease points are sparse

# # Their nearest neighbors are still mostly healthy patients

# # Disease clusters are often small and isolated

# # 3. High Dimensionality Makes It Worse

# # With 50 medical features:

# # Distances between points become less meaningful

# # All patients start to look “similarly far apart”

# # Minority samples lose influence even faster
# # (this is the curse of dimensionality)

# # Simple Intuition (One-Line Explanation)

# # k-NN assumes that “most nearby points belong to the same class” — an assumption that breaks when the class is rare.


In [9]:
# Key Takeaway

# k-NN optimizes for overall similarity, not rare event detection.
# For rare diseases, models like SVM with class weights, logistic regression, anomaly detection, or tree ensembles perform significantly better.